# Exploratory Data Analysis

This notebook provides a lightweight exploratory analysis for the Fashion Product Images (Small) dataset used by the visual search pipeline. It is intended to help inspect dataset quality, metadata coverage, and category balance before feature extraction and evaluation.

## What This Notebook Covers

- Metadata loading and validation
- Dataset size and available columns
- Missing-value analysis
- Category distributions
- Image file availability relative to metadata

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

metadata_path = project_root / 'styles.csv'
images_dir = project_root / 'images'

print(f'Project root: {project_root}')
print(f'Metadata path exists: {metadata_path.exists()}')
print(f'Images directory exists: {images_dir.exists()}')

In [ ]:
if not metadata_path.exists():
    raise FileNotFoundError('styles.csv was not found. Place the dataset in the project root before running this notebook.')

df = pd.read_csv(metadata_path, on_bad_lines='skip')
df['id'] = df['id'].astype(str)

print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns)}')
df.head()

In [ ]:
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'missing_values': df.isna().sum().values,
    'missing_pct': (df.isna().mean() * 100).round(2).values,
    'unique_values': df.nunique(dropna=True).values
})

summary.sort_values('missing_pct', ascending=False).reset_index(drop=True)

## Image Availability Check

The feature extraction pipeline only indexes metadata rows that have a corresponding image file. This check estimates how many metadata records are usable for embedding generation.

In [ ]:
if images_dir.exists():
    image_exists = df['id'].apply(lambda product_id: (images_dir / f'{product_id}.jpg').exists())
    coverage = pd.DataFrame({
        'metric': ['metadata rows', 'available images', 'missing images', 'coverage_pct'],
        'value': [
            len(df),
            int(image_exists.sum()),
            int((~image_exists).sum()),
            round(float(image_exists.mean() * 100), 2)
        ]
    })
    display(coverage)
else:
    print('images/ directory not found. Skipping image coverage check.')

## Category Distributions

These plots are useful for identifying class imbalance and understanding which catalog segments dominate the dataset.

In [ ]:
plot_columns = [column for column in ['masterCategory', 'subCategory', 'articleType', 'gender', 'season', 'usage'] if column in df.columns]
plot_columns

In [ ]:
for column in plot_columns:
    top_counts = df[column].fillna('Unknown').value_counts().head(10).sort_values()
    plt.figure(figsize=(8, 5))
    top_counts.plot(kind='barh', color='#2a6f97')
    plt.title(f'Top 10 {column} values')
    plt.xlabel('Count')
    plt.ylabel(column)
    plt.tight_layout()
    plt.show()

In [ ]:
if 'subCategory' in df.columns:
    df['subCategory'].fillna('Unknown').value_counts().head(15).to_frame('count')
else:
    print('subCategory column not available in metadata.')

## Observations Template

Use this section to document a few short conclusions after running the notebook, for example:

- Which product groups dominate the dataset?
- Are there metadata fields with high missingness?
- Is image coverage close to complete?
- Are there category imbalances that may affect retrieval evaluation?